In [ ]:
# Импорт модулей и библиотек для обработки данных
import pandas as pd
import numpy as np
import json
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# # Импорт модулей и библиотек для решения задачи планирования
from sampo.schemas.schedule import Schedule
from sampo.scheduler.genetic.converter import convert_schedule_to_chromosome
from sampo.scheduler.genetic.converter import convert_chromosome_to_schedule
from sampo.scheduler.genetic.operators import is_chromosome_correct
from sampo.scheduler.genetic.utils import prepare_optimized_data_structures
from sampo.userinput.parser.csv_parser import CSVParser

# Потом можно заменить, если добавить в функциях аргументы по умолчанию
from sampo.schemas.landscape import LandscapeConfiguration
from sampo.schemas.schedule_spec import ScheduleSpec

# fitness functions
from sampo.scheduler.genetic.operators import (
    TimeFitness,
    SumOfResourcesPeaksFitness,
    SumOfResourcesFitness
)
from sampo.utilities.resource_usage import resources_costs_sum

In [ ]:
# Импорт ресурсной модели
from models.field_dev_resources_time_estimator import FieldDevWorkEstimator

## Get project data df

In [ ]:
DATA_PATH = Path('data/csv/gas_network_full_connections.csv')
HISTORY_PATH = Path('data/historical_projects_data.csv')

In [ ]:
project_df = pd.read_csv(DATA_PATH, sep=',')
history_df = pd.read_csv(HISTORY_PATH, sep=';')

In [ ]:
project_fitness_constructor = TimeFitness()
all_connections = True
change_connections = False

In [ ]:
project_df.tail()

## Prepare objects

In [ ]:
project_work_estimator = FieldDevWorkEstimator()

In [ ]:
%%capture
# not print output

wg, contractors = CSVParser.work_graph_and_contractors(
    works_info=CSVParser.read_graph_info(
        project_info=project_df,
        history_data=history_df,
        all_connections=all_connections,
        change_connections_info=change_connections
    ),
    work_resource_estimator=project_work_estimator
)

In [ ]:
class GPNLauncher:

    def __init__(self, wg, contractors, work_estimator):
        self.wg = wg
        self.contractors = contractors
        self.work_estimator = work_estimator
        self.ods = self.get_dict_of_optimized_data_structures(wg, contractors)

        _, self.N_WORKERS, self.N_WORKS = self.ods['resources_border'].shape

    def get_schedule(self, activity_list, resources_matrix, contractor_choices_list=None, ceil_list=None):

        # Default max amount is whatever contractor has
        if ceil_list is None:
            ceil_list = self.ods['contractor_borders']

        # Default is first contractor
        if contractor_choices_list is None:
            contractor_choices_list = np.zeros(self.N_WORKS, dtype=np.int16)
            resources_matrix = np.append(resources_matrix, contractor_choices_list.reshape(-1, 1), axis=1)

        # TODO do we need it?
        spec = ScheduleSpec()
        zones = np.array([])
        
        chromosome = (activity_list, resources_matrix, ceil_list, spec, zones)
        
        if not self.my_is_chromosome_correct(chromosome, self.ods):
            raise Exception("Chromosome seems to be invalid. Please check.")

        node2work, *other_info =  self.my_chromosome_to_schedule(chromosome, self.ods, self.work_estimator)
        scheduled_works = Schedule.from_scheduled_works(node2work.values(), self.wg)

        return (scheduled_works, *other_info)

    def get_makespan(self, activity_list, resources_matrix, contractor_choices_list=None, ceil_list=None):
        scheduled_works, *other_info = self.get_schedule(activity_list, resources_matrix, contractor_choices_list=None, ceil_list=None)
        
        # makespan = scheduled_works.full_schedule_df["finish"].max()
        # return makespan

        # return resources_costs_sum(scheduled_works)

        makespan = scheduled_works.full_schedule_df["finish"].max()
        deadline = 700
        return resources_costs_sum(scheduled_works) * max(1, makespan/deadline)

        

    def get_resource_borders(self, work_id, worker_id):
        min_amount = self.ods['resources_border'][0, worker_id, work_id]
        max_amount = min(
            self.ods['resources_border'][1, worker_id, work_id],
            self.ods['contractor_borders'].max(axis=0)[worker_id]
        )
        return (min_amount, max_amount)     

    # TODO maybe change it later, looks not good
    
    @staticmethod
    def get_dict_of_optimized_data_structures(wg, contractors, landscape=LandscapeConfiguration()):
        ods_tuple = prepare_optimized_data_structures(wg, contractors, landscape)
        ods_names = [
            "worker_pool", "index2node", "index2zone", "work_id2index",
            "worker_name2index", "index2contractor", "worker_pool_indices",
            "contractor2index", "contractor_borders", "node_indices",
            "parents", "children", "resources_border"
        ]
        ods = dict(zip(ods_names, ods_tuple))
        return ods

    @staticmethod
    def my_schedule_to_chromosome(schedule, ods):
        return convert_schedule_to_chromosome(
            ods["work_id2index"], ods["worker_name2index"],
            ods["contractor2index"], ods["contractor_borders"], 
            schedule, ScheduleSpec(), LandscapeConfiguration()
        )
    
    @staticmethod
    def my_chromosome_to_schedule(chromosome, ods, work_estimator):
        return convert_chromosome_to_schedule(
            chromosome,
            ods["worker_pool"], ods["index2node"],
            ods["index2contractor"], ods["index2zone"],
            ods["worker_pool_indices"], ods["worker_name2index"],
            ods["contractor2index"], LandscapeConfiguration(),
            work_estimator=work_estimator
        )

    @staticmethod
    def my_is_chromosome_correct(chromosome, ods):
        return is_chromosome_correct(
            chromosome,
            ods['node_indices'], ods['parents'], ods['contractor_borders']
        )

In [ ]:
class PrecedenceManager:
    """Responsible for managing precedence relations"""
    
    def __init__(self,
                 successors: dict[int, set], 
                 predecessors: dict[int, int]
                 ):
        # number of nodes in a graph
        # not really necessary, but keep it just in case
        self.n_nodes = len(successors)
        # dictionary {node id: set of successors/children}
        self.successors = successors
        # dictionary {node id: number of predecessors}
        # we don't need exact predecessors, just the number is enough
        self.n_predecessors = {key: len(value) for key, value in predecessors.items()}

        # track how many predecessors were completed
        # will be set by the reset method
        self.n_completed_predecessors = None
        # set of jobs that have completed predecessors and were not finished already
        # will be set by the reset method
        self.what_can_start = None 
        # reset the states of nodes
        self.reset_completion()
        

    def reset_completion(self):
        """Clear the info about what jobs were completed"""
        # why use a class with reset(), and not just a function?
        # 1) saving info about successors and predecessors in one place
        # 2) a bit more readable thanks to splitting logic to smaller methods

        # set number of completed predecessors to zero for everyone
        self.n_completed_predecessors = {node_id: 0 for node_id in self.n_predecessors.keys()}
        # at the start, only jobs with no predecessors can start
        self.what_can_start = {node_id for node_id, k in self.n_predecessors.items() if (not k)}


    def set_complete(self, node_index: int):
        """Update state after completing a job"""
        # if job was finished, it cannot be started again
        self.what_can_start.remove(node_index)
        # for each successor of the finished node
        for successor in self.successors[node_index]:
            # now we have one more completed predecessor
            self.n_completed_predecessors[successor] += 1
            # now check if that it was the last predecessors we needed
            if self.n_completed_predecessors[successor] == self.n_predecessors[successor]:
                # if it is, then the job can start
                self.what_can_start.add(successor)


    def get_activity_list(self, priorities_array: list[float]):
        """
        Convert list of priorities for jobs to a valid order
        If priorities_array[A] > priorities_array[B], then
            job A will be added first to order
            (if precedence constraints will allow it)
        """
        # first, reset any previos info
        self.reset_completion()
        # order of jobs added
        activity_list = []
        # now, constructing the list
        for _ in range(self.n_nodes):
            # we add job that is:
            next_to_add = max(
                # 1) has all predecessors finished
                self.what_can_start,
                # 2) has the maximum priority among others
                key=lambda x: priorities_array[x]
            )
            # set selected job as completed
            self.set_complete(next_to_add)
            activity_list.append(next_to_add)
        
        return activity_list

        
    # # TODO can i do this?
    # @staticmethod
    # def remove_self_connections(connections):
    #     return {
    #         key: {element for element in value if element != key}
    #         for key, value in connections.items()
    #     }

In [ ]:
gpn_launcher = GPNLauncher(wg, contractors, project_work_estimator)

In [ ]:
precedence_manager = PrecedenceManager(
    gpn_launcher.ods['children'],
    gpn_launcher.ods['parents']
)

## Check a bit

In [ ]:
N_WORKERS = gpn_launcher.N_WORKERS
N_WORKS = gpn_launcher.N_WORKS

In [ ]:
def relative_values_to_valid(priority_list, resources_fractions):
    activity_list = precedence_manager.get_activity_list(priority_list)

    resource_amounts = np.zeros((N_WORKS, N_WORKERS), dtype=np.int16)
    for work_id in range(N_WORKS):
        for worker_id in range(N_WORKERS):
            min_amount, max_amount = gpn_launcher.get_resource_borders(work_id=work_id, worker_id=worker_id)
            percent = resources_fractions[work_id, worker_id]
            chosen_amount = min_amount + (max_amount - min_amount) * percent
            resource_amounts[work_id, worker_id] = int(chosen_amount)

    return activity_list, resource_amounts

In [ ]:
def get_random_solution_makespan():
    priority_list = np.random.random(N_WORKS)
    resources_percents = np.random.random((N_WORKS, N_WORKERS))
    
    activity_list, resource_amounts = relative_values_to_valid(priority_list, resources_percents)
    t = gpn_launcher.get_makespan(activity_list, resource_amounts)
    return t

In [ ]:
get_random_solution_makespan()

In [ ]:
baseline_random_search = min(get_random_solution_makespan() for _ in range(100))
baseline_random_search

## Evolution

In [ ]:
class EvolutionHistory:

    def __init__(self):
        self.history_best = []
        self.history_mean = []
        self.history_q10 = []

    def record_fitness(self, fitness_array):
        self.history_best.append( np.min(fitness_array) )
        self.history_mean.append( np.mean(fitness_array) )
        self.history_q10.append( np.quantile(fitness_array, 0.10) )

In [ ]:
class AntColony:

    def __init__(self, n_bins_priority=5, n_bins_resources=5, evaporation_rate=0.10):
        self.n_bins_priority = n_bins_priority
        self.n_bins_resources = n_bins_resources

        self.priority_pheromones = 10 * np.ones((N_WORKS, n_bins_priority))
        self.resources_pheromones = 10 * np.ones((N_WORKS, N_WORKERS, n_bins_resources))

        self.evaporation_rate = evaporation_rate
        self.history = EvolutionHistory()
    
    def get_one_solution(self):
        priority_list = [
            np.random.choice(
                np.arange(self.n_bins_priority), 
                p=self.priority_pheromones[work_id] / self.priority_pheromones[work_id].sum()
            )
            for work_id in range(N_WORKS)
        ]
        
        resources_bins = np.zeros((N_WORKS, N_WORKERS), dtype=np.int16)
        resources_percents = np.zeros((N_WORKS, N_WORKERS), dtype=np.float32)
        for work_id in range(N_WORKS):
            for worker_id in range(N_WORKERS):
                
                resource_bin = np.random.choice(
                    np.arange(self.n_bins_resources), 
                    p=self.resources_pheromones[work_id, worker_id] / self.resources_pheromones[work_id, worker_id].sum()
                )
                resources_bins[work_id, worker_id] = int(resource_bin)
                
                resource_fraction = np.linspace(0, 1, self.n_bins_resources)[resource_bin]
                resources_percents[work_id, worker_id] = resource_fraction

        activity_list, resource_amounts = relative_values_to_valid(priority_list, resources_percents)
        t = gpn_launcher.get_makespan(activity_list, resource_amounts)
                
        return np.array(priority_list), np.array(resources_bins), t

    
    def get_generation(self, size=100, sort_by_fitness=True):
        priority_array = []
        resources_array = []
        fitness_array = []

        for _ in range(size):
            priority_list, resources, t = self.get_one_solution()
            priority_array.append(priority_list)
            resources_array.append(resources)
            fitness_array.append(t)

        priority_array = np.array(priority_array)
        resources_array = np.array(resources_array)
        fitness_array = np.array(fitness_array)

        if sort_by_fitness:
            order_by_fitness = np.argsort(fitness_array)
            priority_array = priority_array[order_by_fitness]
            resources_array = resources_array[order_by_fitness]
            fitness_array = fitness_array[order_by_fitness]

        self.history.record_fitness(fitness_array)
        
        return priority_array, resources_array, fitness_array


    def update(self, size=25):
        priority_array, resources_array, fitness_array = self.get_generation(size=size)
        
        total_priority_reward = 0 * self.priority_pheromones
        total_resources_reward = 0 * self.resources_pheromones
        
        for priority_list, resources in zip(priority_array[:5], resources_array[:5]):

            for work_id, value in enumerate(priority_list):
                total_priority_reward[work_id, value] += 100

            for work_id, resources_for_work in enumerate(resources):
                for worker_id, value in enumerate(resources_for_work):
                    try:
                        if worker_id != N_WORKERS:
                            total_resources_reward[work_id, worker_id, value] += 100
                    except:
                        print(work_id, worker_id, value)
        
        self.priority_pheromones = (
            self.priority_pheromones * (1 - self.evaporation_rate) + 
            total_priority_reward * self.evaporation_rate
        )

        self.resources_pheromones = (
            self.resources_pheromones * (1 - self.evaporation_rate) + 
            total_resources_reward * self.evaporation_rate
        )

In [ ]:
colony = AntColony()

In [ ]:
for iteration in range(100):
    colony.update()
    print(iteration, end='\r')

In [ ]:
fig = go.Figure([
    go.Scatter(y=colony.history.history_best, name="Лучшее в поколении", line_color="navy"),
    go.Scatter(y=colony.history.history_mean, name="Среднее в поколении", line_color="green"),
    
    # go.Scatter(y=[baseline_random_search for _ in colony.history.history_best], name="Baseline", line_color="lightgrey"),
    # go.Scatter(y=stats["record_break"], name="Record", line_color="navy", mode="markers"),
    # go.Scatter(y=best_result_line, name="Best Found", line_color="navy")
])

fig.update_layout(height=500, width=1000, template="plotly_white")
fig.update_layout(
    xaxis_title="Поколение",                   
    yaxis_title="Метрика (стоимость + штраф)",
    showlegend=True,
    margin_t=0, margin_b=0, margin_l=0, margin_r=0
)

fig.update_layout(legend=dict(
    yanchor="top", y=0.95,
    xanchor="right", x=0.95
))

fig.show()
# fig.write_image('plots/withlegend_ant.png', scale=4)

In [ ]:
np.min(colony.history.history_best)